In [1]:
%load_ext autoreload
%autoreload 2
from elexon import fetch_mid, fetch_wind
import requests
import pandas as pd
BASE = "https://data.elexon.co.uk/bmrs/api/v1"

In [2]:
mid = fetch_mid("2026-08-01T00:00Z", "2026-10-01T00:00Z")
print(len(mid), mid.index.min(), mid.index.max())
print(mid.index.duplicated().sum())

2926 2026-08-01 00:00:00+00:00 2026-10-01 00:00:00+00:00
0


In [3]:
full = pd.date_range(mid.index.min(), mid.index.max(), freq="30min")
full.difference(mid.index)

DatetimeIndex(['2026-09-11 22:00:00+00:00', '2026-09-11 23:00:00+00:00',
               '2026-09-11 23:30:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)

In [4]:
raw = requests.get(f"{BASE}/balancing/pricing/market-index",
                   params={"from": "2026-09-11T21:00Z", "to": "2026-09-12T00:00Z"},
                   timeout=30).json()["data"]
pd.DataFrame(raw).sort_values("startTime")

,startTime,dataProvider,settlementDate,settlementPeriod,price,volume
12,2026-09-11T21:00:00Z,APXMIDP,2026-09-11,45,176.95,195.35
13,2026-09-11T21:00:00Z,N2EXMIDP,2026-09-11,45,0.00,0.00
10,2026-09-11T21:30:00Z,APXMIDP,2026-09-11,46,171.73,212.65
11,2026-09-11T21:30:00Z,N2EXMIDP,2026-09-11,46,140.00,47.50
8,2026-09-11T22:00:00Z,APXMIDP,2026-09-11,47,0.00,0.00
9,2026-09-11T22:00:00Z,N2EXMIDP,2026-09-11,47,0.00,0.00
6,2026-09-11T22:30:00Z,APXMIDP,2026-09-11,48,0.00,0.00
7,2026-09-11T22:30:00Z,N2EXMIDP,2026-09-11,48,138.84,29.20
4,2026-09-11T23:00:00Z,APXMIDP,2026-09-12,1,0.00,0.00
5,2026-09-11T23:00:00Z,N2EXMIDP,2026-09-12,1,0.00,0.00


In [5]:
raw = requests.get(f"{BASE}/datasets/FUELHH",
                   params={"from": "2026-09-11T21:00Z", "to": "2026-09-12T00:00Z"},
                   timeout=30).json()["data"]
pd.DataFrame(raw).sort_values("startTime")

,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation
539,FUELHH,2026-10-28T00:30:00Z,2026-10-28T00:00:00Z,2026-10-28,1,WIND,11533
520,FUELHH,2026-10-28T00:30:00Z,2026-10-28T00:00:00Z,2026-10-28,1,BIOMASS,3115
521,FUELHH,2026-10-28T00:30:00Z,2026-10-28T00:00:00Z,2026-10-28,1,CCGT,7556
522,FUELHH,2026-10-28T00:30:00Z,2026-10-28T00:00:00Z,2026-10-28,1,COAL,0
523,FUELHH,2026-10-28T00:30:00Z,2026-10-28T00:00:00Z,2026-10-28,1,INTELEC,-1024
...,...,...,...,...,...,...,...
17,FUELHH,2026-10-28T13:30:00Z,2026-10-28T13:00:00Z,2026-10-28,27,OTHER,376
18,FUELHH,2026-10-28T13:30:00Z,2026-10-28T13:00:00Z,2026-10-28,27,PS,244
19,FUELHH,2026-10-28T13:30:00Z,2026-10-28T13:00:00Z,2026-10-28,27,WIND,11979
10,FUELHH,2026-10-28T13:30:00Z,2026-10-28T13:00:00Z,2026-10-28,27,INTNEM,0


In [6]:
wind = fetch_wind("2026-08-01T00:00Z", "2026-10-01T00:00Z")
print(len(wind), wind.index.min(), wind.index.max())
print(wind.index.duplicated().sum())

2976 2026-07-31 23:00:00+00:00 2026-10-01 22:30:00+00:00
0


In [7]:
r = requests.get(f"{BASE}/datasets/FUELHH",
                 params={"settlementDateFrom": "2026-08-01", "settlementDateTo": "2026-08-01"},
                 timeout=30)
print(r.status_code, r.url)
df = pd.DataFrame(r.json()["data"])
print(len(df), df["startTime"].min(), df["startTime"].max())

200 https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH?settlementDateFrom=2026-08-01&settlementDateTo=2026-08-01
960 2026-07-31T23:00:00Z 2026-08-01T22:30:00Z


In [8]:
r = requests.get(f"{BASE}/datasets/FUELHH",
                 params={"settlementDateFrom": "2026-08-01", "settlementDateTo": "2026-09-30",
                         "fuelType": "WIND"}, timeout=30)
print(r.status_code)
print(r.text[:300] if r.status_code != 200 else len(r.json()["data"]))

400
{"type":"https://tools.ietf.org/html/rfc9110#section-15.5.1","title":"One or more validation errors occurred.","status":400,"errors":{"":["The date range between SettlementDateFrom and SettlementDateTo inclusive must not exceed 7 days"]},"traceId":"00-94cb968a3d235281a7e5075f97386152-a72cf2aed83931a


In [10]:
wind = fetch_wind("2026-08-01T00:00Z", "2026-10-01T00:00Z")
print(len(wind), wind.index.min(), wind.index.max(), wind.index.duplicated().sum())
wind["wind_mw"].describe()

2976 2026-07-31 23:00:00+00:00 2026-10-01 22:30:00+00:00 0


count     2976.000000
mean      7321.414987
std       4086.661751
min        398.000000
25%       4219.250000
50%       6554.000000
75%      10438.250000
max      17365.000000
Name: wind_mw, dtype: float64